# Experiment 9: final HISTOPANTUM colorectal ResNet50

Experiment 8 selected the architecture and training recipe using five-fold case-disjoint cross-validation. This notebook trains **one new final model** on all 32 development cases and evaluates it on the same locked 8-case holdout used in Experiment 8. These cases are excluded from Experiment 9 training, but they are not a pristine first-use test set.

Selected from Experiment 8 CV: ResNet50, 2 frozen-head epochs, 3 fine-tuning epochs, and use of partial fine-tuning. Inherited fixed settings: ImageNet initialization, `conv5_*` unfreezing with BatchNorm frozen, learning rates `1e-3`/`1e-5`, batch size 32, seed 42, augmentation/preprocessing, threshold `0.5`, and the existing 32/8 assignment. The holdout results do not select an epoch, checkpoint, threshold, or hyperparameter.

Research and educational use only. This is not clinical validation and must not be used for diagnosis or clinical decisions.

In [ ]:
!unzip /content/histopantum.zip -d /content/histopantum

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from tensorflow import keras

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 2
FINE_TUNE_EPOCHS = 3
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
THRESHOLD = 0.5
DATASET_ROOT = Path(os.environ.get('HISTOPANTUM_COLON_ROOT', '/content/histopantum/histopantum/colon'))
SPLIT_CSV = Path(os.environ.get('HISTOPANTUM_CV_SPLIT_CSV', '/content/cv_split_assignment.csv'))
OUTPUT_DIR = Path('/content/exp9_outputs') if Path('/content').exists() else Path('outputs')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f'Deterministic TensorFlow operations unavailable: {exc}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'python': sys.version, 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__})

{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'tensorflow': '2.20.0', 'sklearn': '1.6.1'}


## Verify the locked case assignment and build the patch manifest

In [ ]:
candidates = [DATASET_ROOT, Path('/content/histopantum/colon'), Path('/content/colon'),
              Path('data/histopantum/colon')]
DATASET_ROOT = next((path for path in candidates if path.is_dir()), DATASET_ROOT)
assert (DATASET_ROOT / 'non-tumour').is_dir() and (DATASET_ROOT / 'tumour').is_dir(), (
    f'Expected non-tumour/ and tumour/ under {DATASET_ROOT}')
assert SPLIT_CSV.is_file(), f'Upload experiments/exp-9/cv_split_assignment.csv: {SPLIT_CSV}'
split = pd.read_csv(SPLIT_CSV)
assert len(split) == 40 and set(split['group']) == {'test', 'fold0', 'fold1', 'fold2', 'fold3', 'fold4'}
assert split.loc[split['group'] == 'test', 'case_id'].nunique() == 8
assert split.loc[split['group'] != 'test', 'case_id'].nunique() == 32
print(split.groupby(split['group'].eq('test').map({False: 'development', True: 'test'})).agg(
    cases=('case_id', 'size'), patches=('patches', 'sum')))

             cases  patches
group                      
development     32    21801
test             8     5447


In [ ]:
def parse_patch(path: Path, label: int) -> dict:
    """Parse one HISTOPANTUM patch filename into auditable identifiers."""
    parts = path.stem.rsplit('_', 2)
    if len(parts) != 3 or not parts[1].isdigit() or not parts[2].isdigit():
        raise ValueError(f'Unexpected patch filename: {path.name}')
    slide_id, x, y = parts
    case_parts = slide_id.split('-')
    if len(case_parts) < 3 or case_parts[0] != 'TCGA':
        raise ValueError(f'Unexpected TCGA slide identifier: {slide_id}')
    return {'path': str(path.resolve()), 'relative_path': path.relative_to(DATASET_ROOT).as_posix(),
            'label': label, 'case_id': '-'.join(case_parts[:3]), 'slide_id': slide_id,
            'x': int(x), 'y': int(y)}

records = []
for class_name, label in [('non-tumour', 0), ('tumour', 1)]:
    paths = sorted((DATASET_ROOT / class_name).glob('*.jpg'))
    assert paths, f'No JPEG files found for {class_name}'
    records.extend(parse_patch(path, label) for path in paths)
manifest = pd.DataFrame(records)
assert len(manifest) == 27248 and manifest['case_id'].nunique() == 40
assert not manifest['relative_path'].duplicated().any() and set(manifest['label']) == {0, 1}
group_of = split.set_index('case_id')['group']
assert set(manifest['case_id']) == set(group_of.index)
manifest['group'] = manifest['case_id'].map(group_of)
recount = manifest.groupby('case_id')['label'].agg(patches='size', tumour='sum')
expected = split.set_index('case_id')[['patches', 'tumour']]
assert recount.equals(expected.loc[recount.index]), 'Dataset differs from the locked assignment.'
development = manifest.loc[manifest['group'] != 'test'].reset_index(drop=True)
test = manifest.loc[manifest['group'] == 'test'].reset_index(drop=True)
assert set(development['case_id']).isdisjoint(test['case_id'])
assert len(development) == 21801 and len(test) == 5447
print({'development_cases': development['case_id'].nunique(), 'development_patches': len(development),
       'test_cases': test['case_id'].nunique(), 'test_patches': len(test)})

{'development_cases': 32, 'development_patches': 21801, 'test_cases': 8, 'test_patches': 5447}


## Build the CV-selected ResNet50 configuration

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path: tf.Tensor, label: tf.Tensor):
    """Decode one JPEG into a fixed-shape float32 RGB tensor."""
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    return tf.cast(image, tf.float32), tf.cast(label, tf.float32)

def make_dataset(frame: pd.DataFrame, training: bool) -> tf.data.Dataset:
    """Create a finite deterministic dataset; augmentation remains in the model."""
    dataset = tf.data.Dataset.from_tensor_slices((frame['path'].to_numpy(), frame['label'].to_numpy()))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def compile_model(model: keras.Model, learning_rate: float) -> None:
    """Compile the binary classifier using the locked optimizer and metrics."""
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss=keras.losses.BinaryCrossentropy(),
                  metrics=[keras.metrics.BinaryAccuracy(name='accuracy'),
                           keras.metrics.Precision(name='precision'),
                           keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='roc_auc')])

augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    keras.layers.RandomRotation(0.25, fill_mode='reflect', seed=SEED),
    keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED),
    keras.layers.RandomContrast(0.10, seed=SEED),
], name='training_augmentation')
backbone = keras.applications.ResNet50(include_top=False, weights='imagenet',
                                        input_shape=(*IMAGE_SIZE, 3), pooling='avg')
backbone.trainable = False
inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
x = augmentation(inputs)
x = keras.layers.Lambda(keras.applications.resnet50.preprocess_input, name='caffe_preprocessing')(x)
x = backbone(x, training=False)
x = keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = keras.layers.Dense(1, activation='sigmoid', name='tumour_probability')(x)
model = keras.Model(inputs, outputs, name='histopantum_crc_final_resnet50')
compile_model(model, HEAD_LEARNING_RATE)
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "histopantum_crc_final_resnet50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ training_augmentation           │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ caffe_preprocessing (Lambda)    │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tumour_probability (Dense)      │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

## Train once on all 32 development cases

There are deliberately no validation callbacks or checkpoint comparisons here. The schedule was fixed from Experiment 8 before this run.

In [6]:
train_ds = make_dataset(development, training=True)
head_history = model.fit(train_ds, epochs=HEAD_EPOCHS, verbose=2)

backbone.trainable = True
for layer in backbone.layers:
    layer.trainable = layer.name.startswith('conv5_') and not isinstance(
        layer, keras.layers.BatchNormalization)
trainable_backbone = [layer.name for layer in backbone.layers if layer.trainable]
assert len(trainable_backbone) == 22, trainable_backbone
assert not any(layer.trainable for layer in backbone.layers
               if isinstance(layer, keras.layers.BatchNormalization))
compile_model(model, FINE_TUNE_LEARNING_RATE)
fine_history = model.fit(train_ds, epochs=FINE_TUNE_EPOCHS, verbose=2)

model_path = OUTPUT_DIR / 'resnet50_final.keras'
model.save(model_path)
history = {'head': head_history.history, 'fine_tuning': fine_history.history}
with (OUTPUT_DIR / 'resnet50_training_history.json').open('w', encoding='utf-8') as handle:
    json.dump(history, handle, indent=2)
print('Saved locked final model:', model_path)

Epoch 1/2
682/682 - 104s - 152ms/step - accuracy: 0.9190 - loss: 0.2106 - precision: 0.9274 - recall: 0.9437 - roc_auc: 0.9707
Epoch 2/2
682/682 - 95s - 140ms/step - accuracy: 0.9519 - loss: 0.1352 - precision: 0.9574 - recall: 0.9656 - roc_auc: 0.9870
Epoch 1/3
682/682 - 141s - 206ms/step - accuracy: 0.9666 - loss: 0.0953 - precision: 0.9713 - recall: 0.9751 - roc_auc: 0.9929
Epoch 2/3
682/682 - 133s - 194ms/step - accuracy: 0.9767 - loss: 0.0649 - precision: 0.9800 - recall: 0.9825 - roc_auc: 0.9964
Epoch 3/3
682/682 - 133s - 195ms/step - accuracy: 0.9816 - loss: 0.0527 - precision: 0.9837 - recall: 0.9868 - roc_auc: 0.9973
Saved locked final model: /content/exp9_outputs/resnet50_final.keras


## Locked reused-holdout evaluation

Run this cell only after training is complete and the model is locked. These eight cases were already evaluated in Experiment 8, so this is a reused internal holdout rather than a pristine final test. Do not use these results to alter the model or rerun alternatives against the same group.

In [7]:
test_ds = make_dataset(test, training=False)
probabilities = model.predict(test_ds, verbose=0).reshape(-1)
y_true = test['label'].to_numpy(dtype=int)
y_pred = (probabilities >= THRESHOLD).astype(int)
assert len(probabilities) == len(test) and np.isfinite(probabilities).all()
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred, zero_division=0),
    'recall': recall_score(y_true, y_pred, zero_division=0),
    'specificity': tn / (tn + fp),
    'f1': f1_score(y_true, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_true, probabilities),
    'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]],
    'threshold': THRESHOLD, 'patches': len(test), 'cases': test['case_id'].nunique(),
}
predictions = test[['relative_path', 'case_id', 'slide_id', 'label']].copy()
predictions['probability'] = probabilities
predictions['prediction'] = y_pred
predictions.to_csv(OUTPUT_DIR / 'resnet50_test_predictions.csv', index=False)
with (OUTPUT_DIR / 'resnet50_test_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)
print(json.dumps(metrics, indent=2))

{
  "accuracy": 0.9217918120066091,
  "balanced_accuracy": 0.8996449875451827,
  "precision": 0.8949053080821553,
  "recall": 0.9905521110126956,
  "specificity": 0.8087378640776699,
  "f1": 0.9403026905829597,
  "roc_auc": 0.9906962945127142,
  "confusion_matrix": [
    [
      1666,
      394
    ],
    [
      32,
      3355
    ]
  ],
  "threshold": 0.5,
  "patches": 5447,
  "cases": 8
}


In [8]:
def sha256_file(path: Path) -> str:
    """Return the SHA-256 digest of one artifact."""
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest_out = {
    'experiment': 9, 'architecture': 'resnet50', 'model_file': model_path.name,
    'model_sha256': sha256_file(model_path), 'model_bytes': model_path.stat().st_size,
    'development_cases': 32, 'development_patches': 21801,
    'test_cases': 8, 'test_patches': 5447, 'threshold': THRESHOLD,
    'selection_source': 'Experiment 8 five-fold case-disjoint cross-validation',
    'head_epochs': HEAD_EPOCHS, 'fine_tune_epochs': FINE_TUNE_EPOCHS,
    'fine_tune_rule': 'conv5_*; BatchNormalization frozen',
    'limitations': ['Only 40 TCGA cases are available.',
                    'Patch observations within a case are correlated.',
                    'Internal evaluation is not external or clinical validation.'],
}
with (OUTPUT_DIR / 'resnet50_model_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(manifest_out, handle, indent=2)
compact = OUTPUT_DIR / 'compact_evidence'
compact.mkdir(exist_ok=True)
for name in ['resnet50_training_history.json', 'resnet50_test_predictions.csv',
             'resnet50_test_metrics.json', 'resnet50_model_manifest.json']:
    shutil.copy2(OUTPUT_DIR / name, compact / name)
archive_base = Path('/content/exp9_compact_evidence') if Path('/content').exists() else Path('exp9_compact_evidence')
archive = Path(shutil.make_archive(str(archive_base), 'zip', compact))
print('Compact evidence:', archive, archive.stat().st_size, 'bytes')
print('Final model:', model_path, model_path.stat().st_size, 'bytes')

Compact evidence: /content/exp9_compact_evidence.zip 47360 bytes
Final model: /content/exp9_outputs/resnet50_final.keras 214693367 bytes


In [9]:
!zip -r /content/exp9_outputs.zip /content/exp9_outputs


  adding: content/exp9_outputs/ (stored 0%)
  adding: content/exp9_outputs/resnet50_model_manifest.json (deflated 43%)
  adding: content/exp9_outputs/resnet50_test_predictions.csv (deflated 91%)
  adding: content/exp9_outputs/compact_evidence/ (stored 0%)
  adding: content/exp9_outputs/compact_evidence/resnet50_model_manifest.json (deflated 43%)
  adding: content/exp9_outputs/compact_evidence/resnet50_test_predictions.csv (deflated 91%)
  adding: content/exp9_outputs/compact_evidence/resnet50_test_metrics.json (deflated 41%)
  adding: content/exp9_outputs/compact_evidence/resnet50_training_history.json (deflated 60%)
  adding: content/exp9_outputs/resnet50_final.keras (deflated 7%)
  adding: content/exp9_outputs/resnet50_test_metrics.json (deflated 41%)
  adding: content/exp9_outputs/resnet50_training_history.json (deflated 60%)
